# 260424 LangChain과 외부 API 통합

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w7_tool_calling/llm_260424_external_api.ipynb)

In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-openai openai

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## 1. 페이지네이션 (Pagination)

**강의 메모**: 어제는 단순 HTTP API 호출로 결과를 가져왔다면, 오늘은 데이터가 많을 때 나눠서 받는 방법.

**왜 필요한가**: API가 천만 건을 한 번에 다 주지 않음. 응답 크기/개수 제한 있음. 크롤링·API 호출 시 분량/시간 조절 없이 가져오면 차단(block) 당한 경험 한 번쯤 있을 것.

**파라미터 키 주의**: `_page`, `_limit` 같은 키는 API마다 다름. 제공자 문서를 보고 맞춰야 함.

**두 가지 방식**:
- **Offset 방식 (page + limit)**: 주소처럼 "0번 페이지 N번째" 식으로 직접 접근. 쉬움. 단, 데이터 변경 시 중복/누락 가능.
- **Cursor 방식**: 다음 커서를 받아 다음 호출 파라미터로 넘김. 실시간 피드(스레드, 트위터)에서 안정적.

**`yield` 활용**: 한 번에 리턴하지 않고 페이지마다 누적. while문에서 아이템이 쌓일 때마다 yield → 호출 측에서 list로 묶음.

## 2. 여러 API 병합 + 마크다운 표 가공

**강의 메모**: 보통 우리가 원하는 데이터가 한 엔드포인트에 딱 가공돼 있는 경우는 거의 없음. 사용자 기본정보(`/users/{id}`), 게시글(`/posts?userId=`), 투두(`/todos?userId=`)를 **순차적으로 호출**해서 합치는 게 기본. 실제 서비스에서도 외부 API + 내부 데이터 조합으로 만듦.

**중첩 필드 접근 (`get_by_path`)**: `address.city`처럼 점(.) 경로로 한 단계씩 들어가는 헬퍼. JSON 계층(hierarchy)이 깊을 때 유용.

**마크다운 표 직접 만들기**: LLM에 "이 dict를 마크다운 표로 바꿔줘" 시키면 ` ``` ` 백틱이 string에 그대로 박히는 등 잘못 나오는 경우가 많음. 이 정도 복잡도는 노가다로 직접 짜는 게 스트레스 덜 받음. 헤더 → 구분선(`---`) → 바디 행 순서로 `|`로 묶음.

**왜 마크다운으로 주는가**: 프롬프트 엔지니어링에서 마크다운 형식으로 데이터를 주면 LLM이 더 잘 파악함.

## 3. 오케스트레이터 + 커스텀 툴 (BaseTool)

**오케스트레이터**: LLM이 지휘자처럼 어떤 툴을 언제·몇 번 부를지 자동으로 판단. 어제 `bind_tools` + `invoke`로 한 게 이미 오케스트레이션. 프로덕션 레벨에선 여기에 **예외 처리, 무한 루프 방지(`max_tool_calls`), 할루시네이션 방지**(존재하지 않는 툴 이름을 LLM이 만들어내는 경우 대비)가 추가됨.

**비유**: 툴이 3개뿐이라 똑똑해 보이지만, 실제론 비슷비슷한 툴이 많아 헷갈려서 잘못 호출하는 일이 잦음.

**Stateless vs Stateful 툴**:
- `@tool` 데코레이터 / `StructuredTool`: **stateless**. A, B 입력 → description 따라 결과만 리턴. 중간 변수/캐시 보존 불가.
- `BaseTool` 상속: **stateful**. 인스턴스 변수, 캐시, 쿼리 로그 등을 내부에 저장 가능. `_run` 메서드(`_` 언더바 주의)에 로직 작성. `invoke` 시 실행됨.

**Pydantic `BaseModel`로 args_schema 지정**: 입력 타입을 강제. `BaseTool`과는 다른 클래스(`pydantic.BaseModel`)임에 주의.